<a href="https://colab.research.google.com/github/avi-dot-ai/FL-W/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/avi-dot-ai/FL-W/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)



## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*



### Finding 1 — *The Anatomy of Growing Content*

The report observes that the growing cohort is longer and younger than the declining cohort (Finding #1, p. 6). A constructive question is: **how was the up/down label assigned, which pages were excluded (for example new, flat, or pages without sufficient comparison data), and does the comparison still hold within each brand and major topic or intent group?** The stated 30-day-versus-previous-30-day definition makes the label legible, but a pooled portfolio comparison can reflect brand mix, age, topic demand, or publication timing. A grouped or within-brand sensitivity check would help show whether the directional pattern transfers beyond the brands contributing the most rows. This asks for stronger scope evidence; it does not turn an observational comparison into a causal claim.

### Finding 4 — *The Freshness Multiplier*

The report observes higher health and impressions for older pages refreshed recently, alongside a stable 31–90-day freshness pattern (Finding #4, p. 9). My methodology question is: **is the update timestamp strictly before every performance window used in the comparison, and are refreshed pages compared with similarly visible, similar-age pages within the same brand rather than with all untouched pages?** Teams may refresh pages already expected to perform well, while health includes visibility inputs such as impressions, position, CTR, and scroll depth. The paper appropriately calls the result observational; a pre/post design with an untreated matched comparison and brand-aware validation would make the decision-support claim more specific.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


Week 5 used a client holdout; this notebook makes the validation gap explicit by running the Week-5-style feature set on a random row split and on an unseen-client split. It then applies a stricter leakage decision: 90-day activity aggregates overlap the 30-day windows that define the label, so they are removed from the final audited model. The comparison separates row-level optimism, client transfer, and overlapping-window risk.

In [2]:
from pathlib import Path
import platform
import numpy as np
import pandas as pd
import json
import os, sys, subprocess

SEED = 42
TOP_KS = (50, 100)

def show(table):
    print(table.to_string(index=False) if isinstance(table, pd.DataFrame) else table)

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Define paths using Path objects, relative to the current working directory (REPO_DIR)
DATA_PATH = Path('data/raw/content_refresh_anonymized.csv')
OUTPUT_DIR = Path('work/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
assert df['content_id'].is_unique
print(f'Rows loaded: {len(df):,}; one row per pseudonymized content item.')
assert df['content_id'].is_unique
df['is_declining_label'] = df['trend_direction'].astype(str).str.lower().eq('down').astype(int)
for c in ('impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d'):
    df[f'log_{c}'] = np.log1p(df[c].clip(lower=0))

NUMERIC = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
CATEGORICAL = ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']
EXCLUDED = {'content_id', 'client_id', 'trend_direction', 'trend_pct', 'is_declining_label', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'provider_used', 'model_used'}
assert set(NUMERIC + CATEGORICAL).isdisjoint(EXCLUDED)
FLAGS = []
for c in NUMERIC:
    flag = f'{c}_missing'
    df[flag] = df[c].isna().astype(int)
    FLAGS.append(flag)
FEATURES = NUMERIC + FLAGS + CATEGORICAL
X, y, groups = df[FEATURES].copy(), df['is_declining_label'].copy(), df['client_id'].copy()

def make_matrices(train, test):
    train_parts, test_parts, sources = [], [], []
    for c in NUMERIC + FLAGS:
        a = pd.to_numeric(train[c], errors='coerce').replace([np.inf, -np.inf], np.nan)
        b = pd.to_numeric(test[c], errors='coerce').replace([np.inf, -np.inf], np.nan)
        median = a.median()
        median = 0.0 if pd.isna(median) else float(median)
        a, b = a.fillna(median).to_numpy(float), b.fillna(median).to_numpy(float)
        mean, std = float(a.mean()), float(a.std())
        std = 1.0 if std == 0 or not np.isfinite(std) else std
        train_parts.append(((a - mean) / std)[:, None])
        test_parts.append(((b - mean) / std)[:, None])
        sources.append(c)
    for c in CATEGORICAL:
        a, b = train[c].fillna('unknown').astype(str), test[c].fillna('unknown').astype(str)
        for value in sorted(a.unique()):
            train_parts.append(a.eq(value).to_numpy(float)[:, None])
            test_parts.append(b.eq(value).to_numpy(float)[:, None])
            sources.append(c)
    return np.hstack(train_parts), np.hstack(test_parts), np.array(sources)

def sigmoid(values):
    return 1 / (1 + np.exp(-np.clip(values, -35, 35)))

def fit_logistic(matrix, target, lr=0.15, l2=0.05, maximum=3000):
    target = np.asarray(target, float)
    weights = np.zeros(matrix.shape[1])
    intercept = float(np.log(target.mean() / (1 - target.mean())))
    for iteration in range(1, maximum + 1):
        residual = sigmoid(matrix @ weights + intercept) - target
        new_weights = weights - lr * ((matrix.T @ residual) / len(target) + l2 * weights)
        new_intercept = intercept - lr * residual.mean()
        if max(np.max(np.abs(new_weights - weights)), abs(new_intercept - intercept)) < 1e-7:
            return new_weights, new_intercept, iteration
        weights, intercept = new_weights, new_intercept
    return weights, intercept, maximum

def probability(matrix, weights, intercept):
    return sigmoid(matrix @ weights + intercept)

def average_precision(actual, scores):
    ordered = np.asarray(actual)[np.argsort(-np.asarray(scores), kind='stable')]
    positives = ordered.sum()
    precision = np.cumsum(ordered) / np.arange(1, len(ordered) + 1)
    return float(precision[ordered == 1].sum() / positives) if positives else 0.0

def roc_auc(actual, scores):
    actual = np.asarray(actual)
    positives, negatives = actual.sum(), len(actual) - actual.sum()
    ranks = pd.Series(scores).rank(method='average').to_numpy()
    return float((ranks[actual == 1].sum() - positives * (positives + 1) / 2) / (positives * negatives))

def metrics(actual, scores, rate):
    ordered = np.asarray(actual)[np.argsort(-np.asarray(scores), kind='stable')]
    result = {'average_precision': average_precision(actual, scores), 'roc_auc': roc_auc(actual, scores)}
    for k in TOP_KS:
        p = float(ordered[:k].mean())
        result.update({f'precision_at_{k}': p, f'positives_at_{k}': int(ordered[:k].sum()), f'lift_at_{k}': p / rate})
    return result

def week4_score(frame):
    eligible = frame['impressions_90d'].ge(3000) & frame['avg_position'].between(4, 10) & frame['ctr'].le(0.30)
    return pd.Series(np.where(eligible, frame['impressions_90d'] * (0.30 - frame['ctr']) / 100, 0.0), index=frame.index)

print(f'Rows: {len(df):,}; clients: {groups.nunique()}; observed decline rate: {y.mean():.1%}.')
print(f'Features: {len(FEATURES)}; seed: {SEED}; pandas {pd.__version__}; NumPy {np.__version__}; Python {platform.python_version()}.')

Rows loaded: 30,000; one row per pseudonymized content item.
Rows: 30,000; clients: 32; observed decline rate: 54.2%.
Features: 44; seed: 42; pandas 2.2.3; NumPy 2.1.3; Python 3.13.15.


In [4]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit

# Define feature sets
W5_NUM = NUMERIC
W5_CAT = CATEGORICAL

# Audited features: remove 90-day activity aggregates due to leakage
AUDIT_NUM = [f for f in NUMERIC if not ('_90d' in f and 'log_' in f)]
AUDIT_CAT = CATEGORICAL # Assuming no categorical leakage from this type of data

# Define a flexible make_matrices function that accepts feature lists
def make_matrices_flex(train_df, test_df, num_features, cat_features):
    train_parts, test_parts, sources = [], [], []

    # Dynamically create flags for the specified numeric features
    current_flags = [f'{c}_missing' for c in num_features]

    features_to_process = num_features + [f for f in current_flags if f in train_df.columns]

    for c in features_to_process:
        a = pd.to_numeric(train_df[c], errors='coerce').replace([np.inf, -np.inf], np.nan)
        b = pd.to_numeric(test_df[c], errors='coerce').replace([np.inf, -np.inf], np.nan)
        median = a.median()
        median = 0.0 if pd.isna(median) else float(median)
        a, b = a.fillna(median).to_numpy(float), b.fillna(median).to_numpy(float)
        mean, std = float(a.mean()), float(a.std())
        std = 1.0 if std == 0 or not np.isfinite(std) else std
        train_parts.append(((a - mean) / std)[:, None])
        test_parts.append(((b - mean) / std)[:, None])
        sources.append(c)
    for c in cat_features:
        a, b = train_df[c].fillna('unknown').astype(str), test_df[c].fillna('unknown').astype(str)
        for value in sorted(a.unique()):
            train_parts.append(a.eq(value).to_numpy(float)[:, None])
            test_parts.append(b.eq(value).to_numpy(float)[:, None])
            sources.append(c)
    return np.hstack(train_parts), np.hstack(test_parts), np.array(sources)

# Define the evaluate function
def evaluate(X_train_df, X_test_df, y_train_s, y_test_s, numeric_cols, categorical_cols):
    X_train_matrix, X_test_matrix, feature_names = make_matrices_flex(X_train_df, X_test_df, numeric_cols, categorical_cols)

    weights, intercept, _ = fit_logistic(X_train_matrix, y_train_s)
    scores = probability(X_test_matrix, weights, intercept)
    rate = y_test_s.mean()
    results = metrics(y_test_s, scores, rate)
    results['base_rate'] = rate
    return results, weights, scores

# Create data splits
# Random split
random_train_X, random_test_X, y_random_train, y_random_test = train_test_split(X, y, test_size=0.3, random_state=SEED)

# Grouped split (unseen clients)
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED)
group_train_idx, group_test_idx = next(gss.split(X, y, groups))

group_train_X = X.iloc[group_train_idx]
group_test_X = X.iloc[group_test_idx]
y_group_train = y.iloc[group_train_idx]
y_group_test = y.iloc[group_test_idx]

random_w5, _, _ = evaluate(random_train_X, random_test_X, y_random_train, y_random_test, W5_NUM, W5_CAT)
group_w5, _, _ = evaluate(group_train_X, group_test_X, y_group_train, y_group_test, W5_NUM, W5_CAT)
group_audit, audited_model, audited_scores = evaluate(group_train_X, group_test_X, y_group_train, y_group_test, AUDIT_NUM, AUDIT_CAT)

comparison = pd.DataFrame([
    {'design': 'Before: random rows, Week-5-style features', **random_w5},
    {'design': 'Honest split: unseen clients, Week-5-style features', **group_w5},
    {'design': 'After: unseen clients, audited metadata-only features', **group_audit},
])
for column in ['base_rate', 'average_precision', 'roc_auc', 'precision_at_50', 'lift_at_50', 'precision_at_100', 'lift_at_100']:
    comparison[column] = comparison[column].round(3)
print('Before/after validation comparison (each metric is computed only on its test partition):')
print(comparison.to_string(index=False))
print('Reading: random-to-grouped is the client-specific optimism gap; the final row also removes windows that overlap the label.')
print('Base rate is printed beside each result so Precision@K is interpreted as lift, not as a standalone percentage.')

Before/after validation comparison (each metric is computed only on its test partition):
                                               design  average_precision  roc_auc  precision_at_50  positives_at_50  lift_at_50  precision_at_100  positives_at_100  lift_at_100  base_rate
           Before: random rows, Week-5-style features              0.722    0.703             0.90               45       1.653              0.87                87        1.598      0.545
  Honest split: unseen clients, Week-5-style features              0.653    0.614             0.80               40       1.430              0.77                77        1.376      0.559
After: unseen clients, audited metadata-only features              0.648    0.607             0.88               44       1.573              0.80                80        1.430      0.559
Reading: random-to-grouped is the client-specific optimism gap; the final row also removes windows that overlap the label.
Base rate is printed beside each res

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*



The target is trend_direction equals down, calculated from the most recent 30 days versus the preceding 30 days. Trend direction, trend percent, and every last-30-day or previous-30-day field are direct label sources and are excluded. The Week-5 90-day totals and rates are not direct copies, but their trailing window contains both label windows; they are therefore treated as overlapping-window features and removed from the final audited model. IDs are grouping-only, and provider/model fields remain excluded because they can encode process decisions rather than page state.

The deliberately leaky probe below is a harness check, not a final model: adding trend direction should make the score nearly perfect because it defines the target.

In [7]:
DIRECT = {'trend_direction', 'trend_pct', 'is_declining_label', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d'}
OVERLAP = {'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'days_with_impressions', 'days_with_sessions'}

# Filter AUDIT_NUM and AUDIT_CAT from their current global state
# These variables (AUDIT_NUM, AUDIT_CAT) are expected to be available from the kernel state,
# previously defined in cell daGQClOC0DHV. This re-filters them based on the OVERLAP set.
AUDIT_NUM = [f for f in AUDIT_NUM if f not in OVERLAP]
AUDIT_CAT = [f for f in AUDIT_CAT if f not in OVERLAP]

final_features = set(AUDIT_NUM + AUDIT_CAT)
assert final_features.isdisjoint(DIRECT | OVERLAP | {'content_id', 'client_id', 'provider_used', 'model_used'})

# Recalculate group_audit with the corrected AUDIT_NUM and AUDIT_CAT
# group_train_X, group_test_X, y_group_train, y_group_test are assumed to be global
# and available from the previous execution of cell daGQClOC0DHV.
group_audit, audited_model, audited_scores = evaluate(group_train_X, group_test_X, y_group_train, y_group_test, AUDIT_NUM, AUDIT_CAT)

# Create temporary dataframes that include 'trend_direction' for the leaky probe
leaky_group_train_X = group_train_X.copy()
leaky_group_test_X = group_test_X.copy()

# Add 'trend_direction' from the original 'df' to these temporary dataframes
# Use .loc to ensure alignment with the indices of group_train_X and group_test_X
leaky_group_train_X['trend_direction'] = df.loc[group_train_X.index, 'trend_direction']
leaky_group_test_X['trend_direction'] = df.loc[group_test_X.index, 'trend_direction']

leaky, _, _ = evaluate(leaky_group_train_X, leaky_group_test_X, y_group_train, y_group_test, AUDIT_NUM, AUDIT_CAT + ['trend_direction'])
audit = pd.DataFrame([
    ['Label derivation', 'trend_direction and trend_pct', 'excluded; target is derived from them'],
    ['Direct label windows', 'last-30-day and previous-30-day fields', 'excluded'],
    ['Overlapping windows', '90-day totals, rates, and visibility counts', 'excluded from final model'],
    ['Identity fields', 'content_id and client_id', 'excluded; client_id used only for grouping'],
    ['Decision-derived fields', 'provider_used and model_used', 'excluded'],
], columns=['risk', 'fields reviewed', 'final decision'])
print(audit.to_string(index=False))
print('Controlled leaky probe — unseen-client AP with trend_direction: {:.3f}; audited final AP: {:.3f}.'.format(leaky['average_precision'], group_audit['average_precision']))
print('Leakage result: PASS — final features contain no direct label source, overlap field, ID, or process field.')
print('Availability limitation: without feature-as-of timestamps, metadata-only results are observed associations, not validated future forecasts.')

                   risk                             fields reviewed                             final decision
       Label derivation               trend_direction and trend_pct      excluded; target is derived from them
   Direct label windows      last-30-day and previous-30-day fields                                   excluded
    Overlapping windows 90-day totals, rates, and visibility counts                  excluded from final model
        Identity fields                    content_id and client_id excluded; client_id used only for grouping
Decision-derived fields                provider_used and model_used                                   excluded
Controlled leaky probe — unseen-client AP with trend_direction: 1.000; audited final AP: 0.612.
Leakage result: PASS — final features contain no direct label source, overlap field, ID, or process field.
Availability limitation: without feature-as-of timestamps, metadata-only results are observed associations, not validated future fo

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


The error review uses the final audited model on unseen clients. The 0.50 cutoff is descriptive only; the review queue is evaluated by ranking metrics above.

**Claim before audit (too broad):** “The model predicts client decline and identifies which pages should be refreshed.”

**Claim after audit:** “On this anonymized snapshot, the metadata-only model measured directional separation of the observed decline label on unseen clients. It may support a review queue alongside human context; it does not forecast future decline or show that refreshing a page will improve performance.”

In [9]:
errors = df.loc[group_test_idx, ['content_type', 'main_intent', 'word_count', 'content_age_days', 'search_volume']].copy()
errors['observed_decline'], errors['model_probability'] = y_group_test, audited_scores
errors['error_type'] = np.select([(errors.model_probability.ge(.5) & errors.observed_decline.eq(0)), (errors.model_probability.lt(.5) & errors.observed_decline.eq(1))], ['false positive', 'false negative'], default='correct')
by_error = errors.loc[errors.error_type.ne('correct')].groupby(['error_type', 'content_type'], dropna=False).size().rename('n_errors').reset_index().sort_values(['error_type', 'n_errors'], ascending=[True, False])
print('Error counts by broad content type (no client or content identifiers):')
print(by_error.to_string(index=False))
examples = errors.loc[errors.error_type.ne('correct'), ['error_type', 'model_probability', 'content_type', 'main_intent', 'word_count', 'content_age_days', 'search_volume']].sort_values(['error_type', 'model_probability'], ascending=[True, False]).head(3).reset_index(drop=True)
examples.insert(0, 'redacted_case', [f'case_{i + 1}' for i in range(len(examples))])
examples['why_review_matters'] = np.where(examples.error_type.eq('false positive'), 'Metadata resembles an observed decline, but recorded trend is not down.', 'Metadata does not resemble a decline, but recorded trend is down.')
print('Three concrete wrong cases; identifiers intentionally removed:')
print(examples.round({'model_probability': 3, 'word_count': 0, 'content_age_days': 0, 'search_volume': 0}).to_string(index=False))
print('At descriptive threshold 0.50: {:,} false positives and {:,} false negatives.'.format((errors.error_type == 'false positive').sum(), (errors.error_type == 'false negative').sum()))
print('Measured takeaway: errors remain substantial, so output is decision-support for review, not an automatic refresh decision.')

Error counts by broad content type (no client or content identifiers):
    error_type       content_type  n_errors
false negative    keyword article       715
false negative comparison article         2
false positive    keyword article      3606
false positive comparison article       298
Three concrete wrong cases; identifiers intentionally removed:
redacted_case     error_type  model_probability    content_type   main_intent  word_count  content_age_days  search_volume                                                why_review_matters
       case_1 false negative                0.5 keyword article transactional      2835.0               187           10.0 Metadata does not resemble a decline, but recorded trend is down.
       case_2 false negative                0.5 keyword article informational      2862.0               487          210.0 Metadata does not resemble a decline, but recorded trend is down.
       case_3 false negative                0.5 keyword article transactional  

## Self-check

- [x] Two paper findings are named and each has a constructive label/validation question.
- [x] A random row split, an unseen-client split, and a stricter audited model are compared with base rates.
- [x] Direct label fields, overlapping windows, IDs, and process fields are audited; a deliberately leaky probe checks the harness.
- [x] Aggregate error patterns and three redacted wrong cases are included.
- [x] Claims use observed, measured, directional, and decision-support language.
- [x] Notebook executed top to bottom in the repository environment; outputs are the record of this audit.